### 6. DECISION TREES AND ENSENBLE LEARNING

In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

df = pd.read_csv('data/CreditScoring.csv', sep=',')

In [22]:
df.columns = df.columns.str.lower().str.replace(' ', '_')
df.head()

,status,seniority,home,time,age,marital,records,job,expenses,income,assets,debt,amount,price
0,1,9,1,60,30,2,1,3,73,129,0,0,800,846
1,1,17,1,60,58,3,1,1,48,131,0,0,1000,1658
2,2,10,2,36,46,2,2,3,90,200,3000,0,2000,2985
3,1,0,1,60,24,1,1,1,63,182,2500,0,900,1325
4,1,0,1,36,26,1,1,1,46,107,0,0,310,910


#### RE-ENCODING THE CATEGORICAL VARIABLES

To handle the categorical variables, we need to address a few important considerations. The R file in the repository provides insights into preprocessing this data.

Here are the key points:

1. Categorical Variable Information: The R file also offers information about the categorical variables. For example:

- ‘Status’ is encoded as ‘good’ (1) and ‘bad’ (2).
- ‘Home’ includes categories like ‘rent,’ ‘owner,’ ‘priv,’ ‘ignore,’ ‘parents,’ and ‘other.’
- ‘Marital’ encompasses ‘single,’ ‘married,’ ‘widow,’ ‘separated,’ and ‘divorced.’
- ‘Records’ has ‘yes’ and ‘no.’
- ‘Job’ includes ‘fixed,’ ‘partime,’ ‘freelance,’ and ‘other.’

2. Missing Values: The missing values are encoded as a series of nines (99999999). We’ll need to address how to handle these missing values.


To proceed, we’ll need to translate these numerical values back into their respective categorical strings. This ensures our data is more interpretable and ready for analysis.

In [23]:
status_values = {
    1: 'ok',
    2: 'default',
    0: 'unk'
}

home_values = {
    1: 'rent',
    2: 'owner',
    3: 'private',
    4: 'ignore',
    5: 'parents',
    6: 'other',
    0: 'unk'
}

marital_values = {
    1: 'single',
    2: 'married',
    3: 'widow',
    4: 'separated',
    5: 'divorced',
    0: 'unk'
}

records_values = {
    1: 'no',
    2: 'yes',
    0: 'unk'
}

job_values = {
    1: 'fixed',
    2: 'partime',
    3: 'freelance',
    4: 'others',
    0: 'unk'
}

df['status'] = df['status'].map(status_values)
df['home'] = df['home'].map(home_values)
df['marital'] = df['marital'].map(marital_values)
df['records'] = df['records'].map(records_values)
df['job'] = df['job'].map(job_values)

In [24]:
df.describe().round().T
# 99999999.0 is a placeholder for missing values in the dataset.
# We will replace it with NaN for better handling of missing data.

,count,mean,std,min,25%,50%,75%,max
seniority,4455.0,8.0,8.0,0.0,2.0,5.0,12.0,48.0
time,4455.0,46.0,15.0,6.0,36.0,48.0,60.0,72.0
age,4455.0,37.0,11.0,18.0,28.0,36.0,45.0,68.0
expenses,4455.0,56.0,20.0,35.0,35.0,51.0,72.0,180.0
income,4455.0,763317.0,8703625.0,0.0,80.0,120.0,166.0,99999999.0
assets,4455.0,1060341.0,10217569.0,0.0,0.0,3500.0,6000.0,99999999.0
debt,4455.0,404382.0,6344253.0,0.0,0.0,0.0,0.0,99999999.0
amount,4455.0,1039.0,475.0,100.0,700.0,1000.0,1300.0,5000.0
price,4455.0,1463.0,628.0,105.0,1118.0,1400.0,1692.0,11140.0


In [25]:
# handling with missing values
replace_values = ['income', 'assets', 'debt']

for col in replace_values:
    df[col] = df[col].replace(99999999.0, np.nan)

# handling with unknown values
# only one row has 'unk' in the 'status' column,
# so we can safely remove it from the dataset.
df = df[df['status'] != 'unk'].reset_index(drop=True)

#### PERFORMING THE DATA SPLIT

In [26]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=11)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=11)

df_full_train.reset_index(drop=True)
df_train.reset_index(drop=True)
df_val.reset_index(drop=True)
df_test.reset_index(drop=True)

y_full_train = (df_full_train['status'] == 'default').astype(int).values
y_train = (df_train['status'] == 'default').astype(int).values
y_val = (df_val['status'] == 'default').astype(int).values
y_test = (df_test['status'] == 'default').astype(int).values

X_full_train = df_full_train.drop(columns=['status'])
X_train = df_train.drop(columns=['status'])
X_val = df_val.drop(columns=['status'])
X_test = df_test.drop(columns=['status'])



#### DECISION TREES

Decision trees are a fundamental machine learning technique used for both classification and regression tasks. Their structure mirrors a flowchart: each internal node represents a feature, each branch a decision rule, and each leaf node a final prediction.

What makes decision trees stand out is their interpretability - unlike many ML models, the logic behind a prediction can be read and understood by humans, making them especially useful when explainability matters.

#### HOW A DECISION TREE IS STRUCTURED

A decision tree is made up of nodes connected by branches. Each node tests a condition on a feature - the result is either true (right branch) or false (left branch). This continues recursively until the tree reaches a leaf node, which holds the final prediction: OK or DEFAULT.

Each path from root to leaf represents a decision rule the model has learned from training data. The deeper the tree, the more specific the rules - though very deep trees risk overfitting.

In [27]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import roc_auc_score

# ## Now we need to turn our training dataframe into a list of dictionaries
# ## then turn this list of dictionaries into the feature matrix. After that we train the model.

train_dict = X_train.fillna(0).to_dict(orient='records')
dv = DictVectorizer(sparse=False)
X_train = dv.fit_transform(train_dict)

val_dict = X_val.fillna(0).to_dict(orient='records')
X_val = dv.transform(val_dict)

In [28]:
visualize = pd.DataFrame(X_train, columns=dv.get_feature_names_out().tolist())
visualize.head()

,age,amount,assets,debt,expenses,home=ignore,home=other,home=owner,home=parents,home=private,...,marital=married,marital=separated,marital=single,marital=unk,marital=widow,price,records=no,records=yes,seniority,time
0,36.0,1000.0,10000.0,0.0,75.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1400.0,1.0,0.0,10.0,36.0
1,32.0,1100.0,0.0,0.0,35.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1330.0,0.0,1.0,6.0,48.0
2,40.0,1320.0,0.0,0.0,75.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1600.0,1.0,0.0,1.0,48.0
3,23.0,1078.0,0.0,0.0,35.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1079.0,1.0,0.0,1.0,48.0
4,46.0,1100.0,4000.0,0.0,60.0,0.0,0.0,1.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1897.0,1.0,0.0,5.0,36.0


In [30]:
max_depth = [2, 3, 4, 5, 10, None]

for max_depth in max_depth:
    tree = DecisionTreeClassifier(max_depth=max_depth, random_state=11)
    tree.fit(X_train, y_train)

    # Validação
    y_pred_val = tree.predict_proba(X_val)[:, 1]
    print(f'max_depth: {max_depth}, ROC_AUC val:   {roc_auc_score(y_val, y_pred_val) * 100:.3f}%')

    # Treino (para checar overfitting)
    y_pred_train = tree.predict_proba(X_train)[:, 1]
    print(f'max_depth: {max_depth}, ROC_AUC train: {roc_auc_score(y_train, y_pred_train) * 100:.3f}%')

    print('---')

max_depth: 2, ROC_AUC val:   66.853%
max_depth: 2, ROC_AUC train: 70.550%
---
max_depth: 3, ROC_AUC val:   73.891%
max_depth: 3, ROC_AUC train: 77.610%
---
max_depth: 4, ROC_AUC val:   76.128%
max_depth: 4, ROC_AUC train: 81.643%
---
max_depth: 5, ROC_AUC val:   76.650%
max_depth: 5, ROC_AUC train: 84.338%
---
max_depth: 10, ROC_AUC val:   69.037%
max_depth: 10, ROC_AUC train: 96.385%
---
max_depth: None, ROC_AUC val:   66.386%
max_depth: None, ROC_AUC train: 100.000%
---


#### Overfitting

Looking at the validation results alone, `max_depth=5` gives the best AUC score of `76.650%` - but compare it with the training score of `84.338%`. That gap is a **warning sign**.

When we let the tree grow freely (max_depth=None), the validation AUC drops to `66.386%` while the training AUC hits a perfect `100%`. This is a textbook case of **overfitting**: the model simply memorizes every single training example, including its noise, and creates rules so specific that they are useless for any new, unseen data. It fails to generalize.

The deeper the tree, the more specific the rules - and the wider the gap between train and validation performance:

| max_depth  | ROC-AUC VAL | ROC-AUC TRAIN |    GAP     |
|:----------:|:-----------:|:-------------:|:----------:|
|      2     |    66.853%  |    70.550%    |    3.7pp   |
|      3     |    73.891%  |    77.610%    |    3.7pp   |
|      4     |    76.128%  |    81.643%    |    5.5pp   |
|      5     |    76.650%  |    84.338%    |    7.7pp   |
|     10     |    69.037%  |    96.385%    |   27.3pp   |
|    None    |    66.386%  |   100.000%    |   33.6pp   |

**By restricting the tree depth, we force it to learn broader**, `more generalizable` rules. `max_depth=5` strikes the best balance here, but it's worth tuning further - for instance by also controlling min_samples_leaf - to reduce that train/val gap without sacrificing too much validation performance.

#### DECISION STUMP

#### DECISION TREE LEARNING ALGORITHM